# Object Detection

Apply your new knowledge of CNNs to one of the hottest (and most challenging!) fields in computer vision: object detection.

**Learning Objectives**
* Identify the components used for object detection (landmark, anchor, bounding box, grid, ...) and their purpose
* Implement object detection
* Implement non-max suppression to increase accuracy
* Implement intersection over union
* Handle bounding boxes, a type of image annotation popular in deep learning
* Apply sparse categorical crossentropy for pixelwise prediction
* Implement semantic image segmentation on the CARLA self-driving car dataset
* Explain the difference between a regular CNN and a U-net
* Build a U-Net

---

## Table of Contents

---

## Object Localization

This section introduces the transition from simple **Image Classification** to **Object Localization** and eventually **Object Detection**. Here is the summary of the key concepts.

* **Classification:** Identifying *what* is in an image (e.g., "this is a car").
* **Classification with Localization:** Identifying the object *and* drawing a bounding box around it (usually for a single object).
* **Detection:** Identifying and localizing *multiple* objects of different categories within the same image.

<div style='display:flex; justify-content:center'>
    <img src='images/class_local_detection.png' width='750px'>
</div>

### Defining the Bounding Box

To localize an object, the network is trained to output four specific real numbers:

* **($b_x, b_y$):** The coordinates of the center point of the object.
* **$b_h$:** The height of the bounding box (expressed as a fraction of total image height).
* **$b_w$:** The width of the bounding box (expressed as a fraction of total image width).
* **Coordinate System:** The top-left of the image is $(0, 0)$ and the bottom-right is $(1, 1)$.

<div style='display:flex; justify-content:center'>
    <img src='images/localization.png' width='750px'>
</div>

### The Target Label Vector ($y$)

In a supervised learning task for localization, the target label $y$ is structured as a multi-component vector (in this example, an 8-element vector):

$$y = [p_c, b_x, b_y, b_h, b_w, c_1, c_2, c_3]^T$$

* **$p_c$ (Probability of Class):** A binary flag where 1 means an object is present and 0 means only background is present.
* **$b_x, b_y, b_h, b_w$:** The bounding box parameters.
* **$c_1, c_2, c_3$:** The class labels (e.g., Pedestrian, Car, Motorcycle).
* **"Don't Cares":** If $p_c=0$, all other elements in the vector are ignored during training.

### The Loss Function

The loss function is calculated differently depending on whether an object is present in the ground truth:

* **If Object Present ($y_1 = 1$):** The loss is typically the sum of squared errors across all 8 components:

$$L(\hat y, y) = \Sigma_{i=1}^8 (\hat y_i - y_i)^2$$

* **If No Object ($y_1 = 0$):** The loss only considers the error in the $p_c$ prediction:

$$L(\hat y, y) = (\hat y_i - y_i)^2$$

> **Note:** While squared error is used for simplicity, specialized losses like **Log Loss** for classes ($c_i$) and $p_c$, or **IoU-based losses** for bounding boxes, are often used in advanced applications.

---

## Landmark Detection

This section explains **Landmark Detection**, a more granular version of localization where the neural network predicts specific coordinates for points of interest rather than just a general bounding box.

### Key Concepts in Landmark Detection

* **Beyond Bounding Boxes:** Instead of just outputting width and height, the network outputs the specific ($x, y$) coordinates for multiple "landmarks" (key points) that define the shape or pose of an object.
* **Vector Construction:** To detect $N$ landmarks, the network outputs a vector $y$ containing:
    * $P_c$: A confidence score (is an object present?).
    * $2N$ coordinates: $(l_{1x}​,l_{1y}​,l_{2x}​,l_{2y}​,\dots, l_{Nx}​,l_{Ny}​)$.
    * *Example:* For 64 facial landmarks, the model would have 129 output units.
* **Consistency is Critical:** For the model to learn, the "identity" of each landmark must be identical across every image in the dataset (e.g., $l_1$ must *always* represent the left corner of the left eye).

<div style='display:flex; justify-content:center'>
    <img src='images/landmark_detection.png' width='750px'>
</div>

### Common Applications

* **Facial Expression & Emotion Recognition:** By tracking landmarks around the mouth and eyes, models can determine if a person is smiling, frowning, or surprised.
* **Augmented Reality (AR) Filters:** Apps like Snapchat use these landmarks to warp faces or accurately place digital objects (like crowns or hats) on a user's head.
* **Human Pose Estimation:** Detecting the "skeleton" of a person by placing landmarks on joints (shoulders, elbows, wrists, knees) to recognize activities or body language.

### Implementation Summary

| Feature | Bounding Box (Localization) | Landmark Detection |
| --- | --- | --- |
| **Output Type** | 4 Numbers ($(b_x​,b_y​,b_h​,b_w​)$) | 2 $\times$ Numbers (Coordinates) |
| **Detail Level** | General position | Specific shape/contour/pose |
| **Data Effort** | Low (Box coordinates) | High (Laborious point annotation) |
| **Model Type** | Regression task | Regression task |

---

## Object Detection

This section introduces the **Sliding Windows Detection Algorithm**, the foundational (though computationally expensive) method for transitioning from simple image classification to full object detection.

### Key Concepts of Sliding Windows

* **Training Phase:**
    * The process begins by training a standard Convolutional Neural Network (ConvNet) on a dataset of *closely cropped* images.
    * The training images ($x$) are cropped so that the object (e.g., a car) is centered and fills most of the frame.
    * The model learns a binary classification ($y$): **1** if a car is present, **0** if not.
* **Detection Phase (The "Sliding" Process):**
    * **Window Selection:** A small rectangular window is chosen and placed over the test image.
    * **Cropping & Classifying:** The region within the window is cropped out and fed into the pre-trained ConvNet to check for the object.
    * **Iteration (Stride):** The window is shifted (using a specific stride) across the entire image, repeating the classification at every step.
    * **Multi-Scale Detection:** To find objects of different sizes, the entire process is repeated multiple times using progressively larger window sizes.

### The Computational Challenge

While conceptually simple, this algorithm has a significant drawback: **High Computational Cost.**

* **The Dilemma:**
    * If you use a *large stride*, you run fewer windows (saving time) but might miss the object or localize it poorly.
    * If you use a *small stride*, you get better accuracy but must run the ConvNet hundreds or thousands of times for a single image.
* **Historical Context:** In the past, sliding windows worked well because classifiers were "cheap" (simple linear functions). Modern ConvNets are "expensive" (deep architectures), making the traditional independent sliding window approach inefficiently slow.

### Summary of the Pipeline

| Stage | Action | Result |
| --- | --- | --- |
| **1. Dataset Prep** | Crop images tightly around the object. | Specialized "Car vs. Not Car" classifier. |
| **2. Initial Pass** | Slide a small window with a fixed stride. | Detects small or distant objects. |
| **3. Scaling** | Increase window size and repeat the slide. | Detects medium and large objects. |
| **4. Classification** | Run each crop through the ConvNet. | Generate a map of object detections. |

---

## Convolutional Implementation of Sliding Windows

This section explains a major breakthrough in object detection efficiency: *Convolutional Implementation of Sliding Windows*. This method allows you to evaluate multiple image regions simultaneously in a single forward pass, rather than cropping and running a network hundreds of times.

### Turning Fully Connected (FC) Layers into Convolutional Layers

To run sliding windows convolutionally, you must first convert the "head" of your network from FC layers to convolutional layers.

* **The Conversion Logic:** A fully connected layer with 400 neurons, for example, can be replaced by a convolutional layer of $1 \times 1 \times 400$ volume using 400 $5 \times 5 \times depth$ filters (assuming the input volume is $5 \times 5 \times depth$). Mathematically, these are identical because each filter "looks" at the entire previous volume, just as a neuron is connected to all previous activations.
* **1x1 Convolutions:** Subsequent FC layers are replaced by $1 \times 1$ convolutional layers, maintaining the $1 \times 1 \times depth$ volume until the final output layer (e.g., $1 \times 1 \times 4$ for 4 classes).

<div style='display:flex; justify-content: center'>
    <img src='images/sliding_win_convnet2.png' width=850px>
</div>

### Convolutional Sliding Windows (The OverFeat Approach)

Once the network is fully convolutional, we can input an image *larger* than the original training size.

* **Shared Computation:** Instead of sliding a window and re-calculating the entire network for every crop, the convolutional layers share computations for overlapping pixels.
* **Output Volumes:** If we train on $14 \times 14$ images and then input a $16 \times 16$ image, the network will naturally output a $2 \times 2 \times 4$ volume instead of a single $1 \times 1 \times 4$ prediction.
* **Spatial Mapping:** Each "pixel" in that $2 \times 2$ output volume corresponds to a specific $14 \times 14$ window in the original image.
    * *Top-Left Pixel:* Result of the top-left $14 \times 14$ window.
    * *Bottom-Right Pixel:* Result of the bottom-right $14 \times 14$ window.

<div style='display:flex; justify-content: center'>
    <img src='images/sliding_win_convnet1.png' width=850px>
</div>


### Efficiency Summary

| | Traditional Sliding Windows | Convolutional Implementation |
| --- | --- | --- |
| **Method** | Crop, resize, and run independently. | Run entire image in one forward pass. |
| **Computation** | Massive duplication of math. | Shares math for overlapping regions. |
| **Speed** | Infeasibly slow for deep ConvNets. | Extremely fast and efficient. |
| **Output** | A single classification per run. | A "heat map" of classifications. |

### Key Takeaways

* **The Stride:** The stride of your sliding window is determined by the **pooling layers** in your network. For instance, a  max pool effectively creates a stride of 2 in the original image.
* **One Pass:** This allows you to process an entire  image and get an  grid of predictions in the same time it would take to do a few independent crops.
* **The Remaining Problem:** While fast, this method still relies on fixed grids, meaning the bounding boxes may not align perfectly with the object.